In [ ]:
!pip uninstall -y torch torchvision torchaudio

In [ ]:
!pip install torch torchvision torchaudio \
  --index-url https://download.pytorch.org/whl/rocm6.4
!pip install sqlmodel
!pip install sqlalchemy

In [4]:
import torch

print("CUDA Available:", torch.cuda.is_available())
print("Device Count:", torch.cuda.device_count())

if torch.cuda.device_count() > 0:
    print("Device Name:", torch.cuda.get_device_name(0))

print("HIP Version:", torch.version.hip)

CUDA Available: True
Device Count: 1
Device Name: AMD Radeon Graphics
HIP Version: 6.4.43484-123eb5128


In [2]:
# ============================================================================
# CELL: DYNAMIC CLASS EXTRACTION & AUTOMATIC YAML GENERATION
# ============================================================================
import os
import yaml  # Built-in or installable via 'pip install pyyaml'
from pathlib import Path

def generate_dynamic_yolo_yaml(val_images_dir, save_yaml_path="dataset.yaml"):
    """
    Scans an image directory, dynamically extracts distinct classes from 
    filenames (splitting on the last underscore), and writes a YOLO-compliant YAML file.
    """
    img_dir = Path(val_images_dir)
    if not img_dir.exists():
        print(f"❌ Error: The directory '{val_images_dir}' does not exist.")
        return None

    # Supported image extensions
    valid_extensions = {'.jpg', '.jpeg', '.png', '.bmp'}
    
    # Track unique class names found in the folder
    discovered_classes = set()
    
    print(f"📂 Scanning directory: {img_dir}")
    for file_path in img_dir.iterdir():
        if file_path.is_file() and file_path.suffix.lower() in valid_extensions:
            filename = file_path.stem  # e.g., 'detergent_added_0016' or 'authentic_0049'
            
            # Split from the right side at the last underscore to detach the number
            if '_' in filename:
                class_prefix = filename.rsplit('_', 1)[0]  # Extracts 'detergent_added' or 'authentic'
                discovered_classes.add(class_prefix)
            else:
                # Fallback if there is no underscore sequence
                discovered_classes.add(filename)

    # Sort classes alphabetically to maintain consistent ordering indexes
    class_list = sorted(list(discovered_classes))
    
    if not class_list:
        print("⚠️ Warning: No valid images found. YAML file creation aborted.")
        return None

    # Create the numeric ID mapping dictionary required by YOLO {0: 'classA', 1: 'classB'}
    class_mapping = {idx: name for idx, name in enumerate(class_list)}
    
    # Establish the absolute root directory path (one level above the images/val folder)
    # Assumes structure: food-guard-ai/data/milk_evaporative_dataset/images/val
    root_path = img_dir.parents[1].as_posix() 
    
    # Construct the structural configuration content
    yaml_data = {
        'path': root_path,
        'train': 'images/train',
        'val': 'images/val',
        'names': class_mapping
    }
    
    # Write directly out to a clean YAML format file
    with open(save_yaml_path, 'w', encoding='utf-8') as yaml_file:
        yaml.dump(yaml_data, yaml_file, default_flow_style=False, sort_keys=False)
        
    print("\n" + "="*60)
    print("🎯 DYNAMIC YAML CONFIGURATION GENERATED SUCCESSFULY!")
    print("="*60)
    print(f"📍 Saved Location : {Path(save_yaml_path).absolute()}")
    print(f"📦 Root Dataset Dir: {root_path}")
    print(f"🏷️ Discovered Classes ({len(class_mapping)}):")
    for cid, cname in class_mapping.items():
        print(f"    👉 Class {cid}: '{cname}'")
    print("="*60 + "\n")
    
    return save_yaml_path


# --- EXECUTE THE GENERATION PIPELINE ---
# Update this path to point exactly to your validation image directory
VAL_FOLDER = r"/workspace/shared/food-guard-ai/data/milk_evaporative_dataset/images/train"
output_yaml = generate_dynamic_yolo_yaml(VAL_FOLDER, save_yaml_path="dataset.yaml")

📂 Scanning directory: /workspace/shared/food-guard-ai/data/milk_evaporative_dataset/images/train

🎯 DYNAMIC YAML CONFIGURATION GENERATED SUCCESSFULY!
📍 Saved Location : /workspace/shared/food-guard-ai/notebooks/dataset.yaml
📦 Root Dataset Dir: /workspace/shared/food-guard-ai/data/milk_evaporative_dataset
🏷️ Discovered Classes (4):
    👉 Class 0: 'authentic'
    👉 Class 1: 'detergent_added'
    👉 Class 2: 'urea_added'
    👉 Class 3: 'water_diluted'



In [3]:
# ============================================================================
# ============================================================================
# FIXED CELL: 12-CLASS YOLO TRAINING (CPU FALLBACK)
# ============================================================================
from ultralytics import YOLO

# Initialize the model structure
model = YOLO('yolov8s.pt') 

yaml_config_path = r"/workspace/shared/food-guard-ai/notebooks/dataset.yaml"


results = model.train(
        data=yaml_config_path,
        epochs=50,         
        imgsz=640,         
        batch=16,          # Set back to 16 since your AMD VRAM can handle it smoothly
        workers=4,         
        project="milk_detection_framework",
        name="yolo12_classes_run"
    )

print("\n" + "="*80)
print("✓ TRAINING PIPELINE RUN COMPLETED!")
print("📂 Best weights are safely saved here:")
print("   'milk_detection_framework/yolo12_classes_run/weights/best.pt'")
print("="*80)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.66 🚀 Python-3.12.11 torch-2.9.1+rocm6.4 CUDA:0 (AMD Radeon Graphics, 196592MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/workspace/shared/food-guard-ai/notebooks/dataset.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False

/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory


Overriding model.yaml nc=80 with nc=4

                   from  n    params  module                                       arguments                     
  0                  -1  1       928  ultralytics.nn.modules.conv.Conv             [3, 32, 3, 2]                 
  1                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  2                  -1  1     29056  ultralytics.nn.modules.block.C2f             [64, 64, 1, True]             
  3                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  4                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  5                  -1  1    295424  ultralytics.nn.modules.conv.Conv             [128, 256, 3, 2]              
  6                  -1  2    788480  ultralytics.nn.modules.block.C2f             [256, 256, 2, True]           
  7                  -1  1   1180672  ultralytics

/usr/local/lib/python3.12/dist-packages/ultralytics/nn/modules/block.py:1324: UserWarning: Unmatched validator: "HIP_VERSION" is required, but the tuning results does not provide it.  (Triggered internally at /pytorch/aten/src/ATen/hip/tunable/Tunable.cpp:329.)
  attn = (q.transpose(-2, -1) @ k) * self.scale
/usr/local/lib/python3.12/dist-packages/ultralytics/nn/modules/block.py:1324: UserWarning: Unmatched validator: "ROCM_VERSION" is provided, but pytorch is unable to consume it.  (Triggered internally at /pytorch/aten/src/ATen/hip/tunable/Tunable.cpp:337.)
  attn = (q.transpose(-2, -1) @ k) * self.scale


AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 185.5±112.9 MB/s, size: 75.8 KB)
train: Scanning /workspace/shared/food-guard-ai/data/milk_evaporative_dataset/labels/train.cache... 800 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 800/800 115.7Mit/s 0.0s
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 143.9±74.2 MB/s, size: 61.1 KB)
val: Scanning /workspace/shared/food-guard-ai/data/milk_evaporative_dataset/labels/val.cache... 200 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 200/200 3.5Mit/s 0.0s
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.00125, momentum=0.9) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)
Plotting labels to /workspace/shared/food-guard-ai/notebooks/runs/detect/milk_detection_framework/yolo12_classes_run-3/labels.jpg... 
Image sizes 640 train, 640 val
Using 4 datal

In [ ]:
CREATE TABLE vision_nalysis  (
    data_sample_id VARCHAR(36) PRIMARY KEY,
    image_filename VARCHAR(255) NOT NULL,
    
    -- The single string column holding the entire packed JSON payload
    class_counts_json TEXT NOT NULL, 
    
    -- Summary Metrics
    total_objects INT NOT NULL DEFAULT 0,
    unique_classes TEXT NOT NULL, 
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);


shared/food-guard-ai/data/milk_evaporative_dataset/images/val/authentic_0200.jpg

In [9]:
import json
import uuid
from collections import Counter
from pathlib import Path
from datetime import datetime
from ultralytics import YOLO
from sqlmodel import Session
from sqlmodel import SQLModel
from sqlmodel import Field
from sqlmodel import create_engine

class VisionPredictor:
    def __init__(self, weights_path="milk_detection_framework/yolo12_classes_run/weights/best.pt"):
        """
        Loads the core YOLO weights and prepares internal class indexing maps.
        """
        print("⚙️ Loading weights into YOLOv12 Core...")
        self.model = YOLO(weights_path)
        
        # Isolate the 12 ordered target categories
        self.all_classes = [self.model.names[i] for i in sorted(self.model.names.keys())]

    def analyze_image(self, session: Session, image_path: str, batch_id: str, conf_thresh:0.25) -> VisionAnalysis:
        """
        Predicts objects, extracts flat counts, structures a stringified 
        JSON payload, and saves a VisionAnalysis entry via SQLModel Session.
        """
        img_file = Path(image_path)
        if not img_file.exists():
            raise FileNotFoundError(f"Target milk frame missing at: {image_path}")

        # 1. Execute object detection forward pass
        results = self.model.predict(source=str(img_file), conf=conf_thresh, device='cpu', verbose=False)
        result = results[0]

        # 2. Extract detected labels
        detected_labels = []
        highest_score = 0.0
        
        if len(result.boxes) > 0:
            # Capture the top confidence score found across all frames
            highest_score = float(result.boxes.conf.max().item())
            for box in result.boxes:
                class_id = int(box.cls[0].item())
                detected_labels.append(self.model.names[class_id])
        else:
            # Fallback if the sample contains no anomalous objects
            detected_labels.append("authentic")
            highest_score = 1.0  # Perfect confidence that it is clean milk

        # 3. Create the 12-class flattened counts dictionary payload
        counts_tally = Counter(detected_labels)
        payload_dict = {cname: counts_tally.get(cname, 0) for cname in self.all_classes}
        
        # Serialize your 12-class dictionary into a clean, flat string column structure
        stringified_findings_json = json.dumps(payload_dict)

        # Determine the primary winning classification tag for the 'label' field
        top_prediction_label = counts_tally.most_common(1)[0][0]

        # 4. Construct the complete VisionAnalysis SQLModel record instance
        analysis_record = VisionAnalysis(
            id=str(uuid.uuid4())[:18],
            batch_id=batch_id,
            image_path=str(img_file.absolute()),
            model_name="YOLOv12",
            score=round(highest_score, 4),
            label=top_prediction_label,
            total_objects=len(result.boxes) if len(result.boxes) > 0 else 1,
            findings=stringified_findings_json, # <-- Flat text data insertion
            created_at=datetime.utcnow()
        )

        # 5. Insert directly using the active SQLModel Session unit of work
        session.add(analysis_record)
        session.commit()
        session.refresh(analysis_record)

        return analysis_record

NameError: name 'VisionAnalysis' is not defined

In [4]:
# ============================================================================
# PRODUCTION SCRIPT: STANDALONE YOLO TO JSON EXPORT PIPELINE
# ============================================================================
import os
import json
from pathlib import Path
from datetime import datetime
from collections import Counter
from ultralytics import YOLO


class MilkYOLOPredictor:

    def __init__(
        self,
        weights_path="/workspace/shared/food-guard-ai/notebooks/runs/detect/milk_detection_framework/yolo12_classes_run-3/weights/best.pt",
        output_dir="output/yolo_predictions"
    ):
        """
        Initializes the model, extracts class names, and sets up the output directory.
        """
        print("🚀 Loading YOLO model architecture...")
        self.model = YOLO(weights_path)
        
        # Pull and sort all 12 target categories alphabetically from your model config
        self.all_classes = sorted([v for k, v in self.model.names.items()])
        
        # Establish the target JSON file folder location
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        
        print("🏷️ Initialized Master 12-Class Registry:")
        print(self.all_classes)
        print(f"📂 JSON export directory ready at: {self.output_dir.absolute()}\n")

    def predict_folder(self, image_folder, conf=0.25):
        """
        Loops through an input directory to process all valid image frames.
        """
        image_folder = Path(image_folder)
        if not image_folder.exists():
            raise FileNotFoundError(f"Target folder not found: {image_folder}")

        valid_ext = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
        total = 0

        for image_file in image_folder.iterdir():
            if image_file.suffix.lower() not in valid_ext:
                continue

            self.predict_image(image_path=str(image_file), conf=conf)
            total += 1

        print(f"\n✓ Complete! Exported {total} JSON result files to disk.")

    def predict_image(self, image_path, conf=0.25):
        """
        Runs prediction on a single image file and builds the strict 12-class JSON output.
        """
        image_file = Path(image_path)
        print(f"🔍 Analyzing Frame: {image_file.name}")

        # Run inference using standard CPU execution for system stability
        results = self.model.predict(source=str(image_file), conf=conf, device='cpu', verbose=False)
        result = results[0]

        detected_labels = []
        max_conf = 0.0

        # Process bounding boxes ONLY if the model actually detects them
        if len(result.boxes) > 0:
            max_conf = float(result.boxes.conf.max().item())
            for box in result.boxes:
                cls_id = int(box.cls[0].item())
                detected_labels.append(self.model.names[cls_id])
        else:
            max_conf = 0.0

        # Tally occurrences from our detection bucket
        found_counts = Counter(detected_labels)
        
        # Build the exact 12-class dictionary payload (unseen classes auto-default to 0)
        full_12_class_counts = {cname: found_counts.get(cname, 0) for cname in self.all_classes}

        # Structure metadata tokens according to your exact mapping requirements
        sample_id = image_file.stem                  # <-- CHANGED: Now returns "authentic_0246" instead of "authentic_0246.jpg"
        absolute_image_path = str(image_file.resolve()) # <-- image_filename remains the absolute path
        
        total_objects = len(detected_labels)
        unique_classes_list = sorted(list(set(detected_labels)))
        timestamp_now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        # Assemble the finalized structured dataset record
        payload = {
            "data_sample_id": sample_id,
            "image_filename": absolute_image_path,
            "class_counts_json": full_12_class_counts,  # True native dense dictionary mapping
            "total_objects": total_objects,
            "unique_classes": unique_classes_list,      # Clean JSON array tracking
            "created_at": timestamp_now
        }

        # Save to file using the clean name as the tracking filename (authentic_0246.json)
        self.save_json(payload, filename=sample_id)
        
        print(f"   ↳ ID (Name)   : {sample_id}")
        print(f"   ↳ File Path   : {absolute_image_path}")
        print(f"   ↳ Objects     : {total_objects} | Score: {round(max_conf, 4)} | Classes: {unique_classes_list}")

    def save_json(self, payload, filename):
        """
        Writes the prediction payload map directly to an individual file on disk.
        """
        # Saves as e.g., output/yolo_predictions/authentic_0246.json
        json_path = self.output_dir / f"{filename}.json"
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=4)  # Generates clean, human-readable JSON formats

In [5]:
# ============================================================================
# HOW TO RUN THE SCRIPT
# ============================================================================
if __name__ == "__main__":
    
    # Define paths according to your environment layout
    MODEL_WEIGHTS = "/workspace/shared/food-guard-ai/notebooks/runs/detect/milk_detection_framework/yolo12_classes_run-3/weights/best.pt"
    INPUT_VAL_FOLDER = "/workspace/shared/food-guard-ai/data/milk_evaporative_dataset/images/val/"
    JSON_OUTPUT_FOLDER = "output/yolo_predictions"
    
    # Initialize the engine
    predictor = MilkYOLOPredictor(
        weights_path=MODEL_WEIGHTS,
        output_dir=JSON_OUTPUT_FOLDER
    )
    
    # Execute batch folder processing loop safely within the catch architecture
    try:
        predictor.predict_folder(image_folder=INPUT_VAL_FOLDER, conf=0.25)
    except Exception as e:
        print(f"❌ Execution stopped with an operational error: {e}")

🚀 Loading YOLO model architecture...
🏷️ Initialized Master 12-Class Registry:
['authentic', 'detergent_added', 'urea_added', 'water_diluted']
📂 JSON export directory ready at: /workspace/shared/food-guard-ai/notebooks/output/yolo_predictions

🔍 Analyzing Frame: detergent_added_0221.jpg
   ↳ ID (Name)   : detergent_added_0221
   ↳ File Path   : /workspace/shared/food-guard-ai/data/milk_evaporative_dataset/images/val/detergent_added_0221.jpg
   ↳ Objects     : 1 | Score: 0.9873 | Classes: ['water_diluted']
🔍 Analyzing Frame: water_diluted_0229.jpg
   ↳ ID (Name)   : water_diluted_0229
   ↳ File Path   : /workspace/shared/food-guard-ai/data/milk_evaporative_dataset/images/val/water_diluted_0229.jpg
   ↳ Objects     : 1 | Score: 0.9887 | Classes: ['detergent_added']
🔍 Analyzing Frame: authentic_0223.jpg
   ↳ ID (Name)   : authentic_0223
   ↳ File Path   : /workspace/shared/food-guard-ai/data/milk_evaporative_dataset/images/val/authentic_0223.jpg
   ↳ Objects     : 1 | Score: 0.9924 | Class

In [11]:
print (DB_PATH)

data/foodguard.db


In [1]:
# ============================================================================
# PRODUCTION SCRIPT: STANDALONE YOLO TO JSON EXPORT PIPELINE
# ============================================================================
import os
import json
from pathlib import Path
from datetime import datetime
from collections import Counter
from ultralytics import YOLO


class MilkYOLOPredictor:

    def __init__(
        self,
        weights_path="/workspace/shared/food-guard-ai/notebooks/runs/detect/milk_detection_framework/yolo12_classes_run-3/weights/best.pt",
        output_dir="output/yolo_predictions"
    ):
        """
        Initializes the model, extracts class names, and sets up the output directory.
        """
        print("🚀 Loading YOLO model architecture...")
        self.model = YOLO(weights_path)
        
        # Pull and sort all 12 target categories alphabetically from your model config
        self.all_classes = sorted([v for k, v in self.model.names.items()])
        
        # Establish the target JSON file folder location
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        
        print("🏷️ Initialized Master 12-Class Registry:")
        print(self.all_classes)
        print(f"📂 JSON export directory ready at: {self.output_dir.absolute()}\n")

    def predict_folder(self, image_folder, conf=0.25):
        """
        Loops through an input directory to process all valid image frames.
        """
        image_folder = Path(image_folder)
        if not image_folder.exists():
            raise FileNotFoundError(f"Target folder not found: {image_folder}")

        valid_ext = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
        total = 0

        for image_file in image_folder.iterdir():
            if image_file.suffix.lower() not in valid_ext:
                continue

            self.predict_image(image_path=str(image_file), conf=conf)
            total += 1

        print(f"\n✓ Complete! Exported {total} JSON result files to disk.")

    def predict_image(self, image_path, conf=0.25):
        """
        Runs prediction on a single image file and builds the strict 12-class JSON output.
        """
        image_file = Path(image_path)
        print(f"🔍 Analyzing Frame: {image_file.name}")

        # Run inference using standard CPU execution for system stability
        results = self.model.predict(source=str(image_file), conf=conf, device='cpu', verbose=False)
        result = results[0]

        detected_labels = []
        max_conf = 0.0

        # Process bounding boxes ONLY if the model actually detects them
        if len(result.boxes) > 0:
            max_conf = float(result.boxes.conf.max().item())
            for box in result.boxes:
                cls_id = int(box.cls[0].item())
                detected_labels.append(self.model.names[cls_id])
        else:
            max_conf = 0.0

        # Tally occurrences from our detection bucket
        found_counts = Counter(detected_labels)
        
        # Build the exact 12-class dictionary payload (unseen classes auto-default to 0)
        full_12_class_counts = {cname: found_counts.get(cname, 0) for cname in self.all_classes}

        # Structure metadata tokens according to your swapped mapping requirements
        sample_id = image_file.name                  # <-- data_sample_id is now the filename
        absolute_image_path = str(image_file.resolve()) # <-- image_filename is now the absolute path
        
        total_objects = len(detected_labels)
        unique_classes_list = sorted(list(set(detected_labels)))
        timestamp_now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        # Assemble the finalized structured dataset record
        payload = {
            "data_sample_id": sample_id,
            "image_filename": absolute_image_path,
            "class_counts_json": full_12_class_counts,  # True native dense dictionary mapping
            "total_objects": total_objects,
            "unique_classes": unique_classes_list,      # Clean JSON array tracking
            "created_at": timestamp_now
        }

        # Save to file using the image filename as the base tracking name
        self.save_json(payload, filename=image_file.name)
        
        print(f"   ↳ ID (Name)   : {sample_id}")
        print(f"   ↳ File Path   : {absolute_image_path}")
        print(f"   ↳ Objects     : {total_objects} | Score: {round(max_conf, 4)} | Classes: {unique_classes_list}")

    def save_json(self, payload, filename):
        """
        Writes the prediction payload map directly to an individual file on disk.
        """
        # Saves as e.g., output/yolo_predictions/detergent_added_0016.jpg.json
        json_path = self.output_dir / f"{filename}.json"
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=4)  # Generates clean, human-readable JSON formats



In [3]:
if __name__ == "__main__":
    
    # Define paths according to your workspace environment layout
    MODEL_WEIGHTS = "/workspace/shared/food-guard-ai/notebooks/runs/detect/milk_detection_framework/yolo12_classes_run-3/weights/best.pt"
    INPUT_VAL_FOLDER = "/workspace/shared/food-guard-ai/data/milk_evaporative_dataset/images/val/"
    JSON_OUTPUT_FOLDER = "output/yolo_predictions"
    
    # Initialize the engine
    predictor = MilkYOLOPredictor( weights_path=MODEL_WEIGHTS, 
        output_dir=JSON_OUTPUT_FOLDER
    )
    
    # Execute batch folder processing loop safely within the catch architecture
    try:
        predictor.predict_folder(image_folder=INPUT_VAL_FOLDER, conf=0.25)
    except Exception as e:
        print(f"❌ Execution stopped with an operational error: {e}")

🚀 Loading YOLO model architecture...
🏷️ Initialized Master 12-Class Registry:
['authentic', 'detergent_added', 'urea_added', 'water_diluted']
📂 JSON export directory ready at: /workspace/shared/food-guard-ai/notebooks/output/yolo_predictions

🔍 Analyzing Frame: detergent_added_0221.jpg


/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory


   ↳ ID (Name)   : detergent_added_0221.jpg
   ↳ File Path   : /workspace/shared/food-guard-ai/data/milk_evaporative_dataset/images/val/detergent_added_0221.jpg
   ↳ Objects     : 1 | Score: 0.9873 | Classes: ['water_diluted']
🔍 Analyzing Frame: water_diluted_0229.jpg
   ↳ ID (Name)   : water_diluted_0229.jpg
   ↳ File Path   : /workspace/shared/food-guard-ai/data/milk_evaporative_dataset/images/val/water_diluted_0229.jpg
   ↳ Objects     : 1 | Score: 0.9887 | Classes: ['detergent_added']
🔍 Analyzing Frame: authentic_0223.jpg
   ↳ ID (Name)   : authentic_0223.jpg
   ↳ File Path   : /workspace/shared/food-guard-ai/data/milk_evaporative_dataset/images/val/authentic_0223.jpg
   ↳ Objects     : 1 | Score: 0.9924 | Classes: ['authentic']
🔍 Analyzing Frame: urea_added_0245.jpg
   ↳ ID (Name)   : urea_added_0245.jpg
   ↳ File Path   : /workspace/shared/food-guard-ai/data/milk_evaporative_dataset/images/val/urea_added_0245.jpg
   ↳ Objects     : 1 | Score: 0.9885 | Classes: ['urea_added']
🔍 An